# 032 — Lógica difusa y control aproximado

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Soluciones explicadas

**E1.** A 12 °C: `μ_frío = (20−12)/10 = 0.8`, `μ_templado = (12−10)/10 = 0.2`. A 18 °C: `μ_frío = 0.2`, `μ_templado = 0.8`. Ambas valen 0.5 en `x = 15` (cruce simétrico de los triángulos): la frontera lingüística entre frío y templado.

**E2.** Fuerzas: `f1 = 0.8`, `f2 = 0.2`. Salida `= (0.8·80 + 0.2·40)/(0.8+0.2) = 72/1 = 72`. La regla dominante manda, pero la minoritaria suaviza el resultado.

**E3.** Nítido: 14 °C → 80; 15 °C → 40; 16 °C → 40 (salto brusco de 40 unidades por 1 grado). Difuso: 14 °C → `(0.6·80+0.4·40)/1 = 64`; 15 °C → `(0.5·80+0.5·40) = 60`; 16 °C → `(0.4·80+0.6·40) = 56`. El controlador difuso **interpola**: sin discontinuidades en el actuador, menos oscilación alrededor del umbral.

**E4.** `μ_¬frío(12) = 1−0.8 = 0.2`; `μ_frío∧templado(12) = min(0.8, 0.2) = 0.2`; `μ_frío∨templado(12) = max(0.8, 0.2) = 0.8`. Nota que AND no da 0: a 12 °C es *un poco* verdad que hace frío y templado a la vez — vaguedad, no contradicción.


In [ ]:
result = run_lab("logic", seed=32)
assert result["kind"] == "logic"
assert result["evidence"]
show(result)


In [ ]:
def tri(x, a, b, c):
    if x <= a or x >= c: return 0.0
    return (x-a)/(b-a) if x <= b else (c-x)/(c-b)

frio = lambda x: tri(x, 0, 10, 20)
templado = lambda x: tri(x, 10, 20, 30)

print("E1:", frio(12), templado(12), "|", frio(18), templado(18))
def salida(x):
    f1, f2 = frio(x), templado(x)
    return (f1*80 + f2*40)/(f1+f2)
print("E2:", salida(12))
for x in (14, 15, 16):
    nitido = 80 if x < 15 else 40
    print(f"E3 x={x}: nitido={nitido} difuso={salida(x):.0f}")
print("E4:", 1-frio(12), min(frio(12), templado(12)), max(frio(12), templado(12)))


## Reflexión

1. `μ_alto(x) = 0.6` y `P(alto) = 0.6` usan el mismo número para cosas distintas. Explica la diferencia con un ejemplo donde confundirlas lleve a una conclusión absurda.
2. En lógica difusa `μ_A∧¬A(x) = min(μ, 1−μ)` puede ser 0.5, no 0. ¿Qué principio de la lógica clásica se pierde y por qué eso es aceptable para predicados vagos?
3. ¿Qué aporta la defuzzificación por centroide frente a elegir la regla de mayor activación? ¿Cuándo darían resultados muy distintos?
